In [ ]:
# Edit only attached Kaggle Input paths and batch sizes; the comparison is frozen.
from pathlib import Path

LEGALIR_SOURCE_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/train.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/selected-contexts")
DENSE_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/bge-m3-kaggle")
RERANKER_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/bge-reranker-v2-m3-kaggle")

CORPUS_BATCH_SIZE = 256  # May be reduced for OOM; semantics do not change.
QUERY_BATCH_SIZE = 64    # May be reduced for OOM; semantics do not change.
RERANKER_BATCH_SIZE = 128  # May only be reduced for OOM.

DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DECLARED_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_DECLARED_REVISION = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
EXPECTED_SOURCE_SHA256 = "c39cde9e74977e350f1456e7d487aafe67d2bcbaa4fa26fcabd557fe635635b7"
EXPECTED_SPLIT_COUNTS = {"train": 4_941, "dev": 1_036, "holdout": 1_023}
EXPECTED_DOCUMENTS = 8_532
EXPECTED_CHUNKS = 199_816
EXPECTED_DENSE_RECALL_AT_100 = 0.9757249918540242
EXPECTED_M2_PRECISION = 0.18338220918866083
EXPECTED_M2_RECALL = 0.8659986966438579
EXPECTED_M2_MRR = 0.7299178380421532

CHUNK_SIZE = 2_000
CHUNK_OVERLAP = 200
CHUNK_STEP = 1_800
TOP_K_CHUNKS = 2_000
DOCUMENT_AGGREGATION = "sum_top_2_dense_chunk_scores"
CANDIDATE_DEPTH = 100
SUPPORT_POOL_SIZES = (2, 8)
CE_SELECTED_CHUNKS = 2
DENSE_MAX_LENGTH = 8_192
RERANKER_MAX_SEQUENCE_LENGTH = 8_192
FINAL_K = 5
SANITY_ABS_TOLERANCE = 1e-9
RESULT_PATH = Path(
    "/kaggle/working/dense_cross_encoder_supporting_evidence_holdout_results.json"
)


In [ ]:
# Enforce Internet-OFF holdout execution and fail loudly for missing attached artifacts.
import os

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

for path, description, must_be_directory in (
    (LEGALIR_SOURCE_PATH, "LegalIR source JSON", False),
    (CORPUS_PATH, "LegalIR corpus directory", True),
    (DENSE_MODEL_PATH, "complete local BGE-M3 snapshot", True),
    (RERANKER_MODEL_PATH, "complete local BGE reranker snapshot", True),
):
    exists = path.is_dir() if must_be_directory else path.is_file()
    if not exists:
        raise FileNotFoundError(f"Attach the {description} at: {path}")


In [ ]:
# Standalone aggregate-only fixed-local-holdout implementation.
import gc
import json
from collections import Counter, defaultdict
from hashlib import sha256
from math import isfinite
from statistics import median
from time import perf_counter

import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer


def read_json(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_fixed_holdout(path: Path) -> tuple[dict, dict]:
    source_hash = sha256(path.read_bytes()).hexdigest()
    if source_hash != EXPECTED_SOURCE_SHA256:
        raise RuntimeError(
            f"LegalIR source SHA-256 mismatch: expected {EXPECTED_SOURCE_SHA256}, "
            f"got {source_hash}. Stop before reading samples."
        )
    value = read_json(path)
    if not isinstance(value, dict) or not all(isinstance(v, dict) for v in value.values()):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    samples = {str(sample_id): sample for sample_id, sample in value.items()}
    if len(samples) != len(value):
        raise ValueError("duplicate sample IDs after string canonicalization")

    counts = {"train": 0, "dev": 0, "holdout": 0}
    holdout_ids = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        group_key = question if isinstance(question, str) else f"\0fallback-sample-id:{sample_id}"
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        split_name = "train" if bucket < 70 else "dev" if bucket < 85 else "holdout"
        counts[split_name] += 1
        if split_name == "holdout":
            holdout_ids.append(sample_id)
    if counts != EXPECTED_SPLIT_COUNTS:
        raise RuntimeError(
            f"fixed split counts mismatch: {counts}. Stop; do not evaluate any split."
        )

    holdout_ids.sort()
    holdout = {sample_id: samples[sample_id] for sample_id in holdout_ids}
    del samples, value
    for sample_id, sample in holdout.items():
        if not isinstance(sample.get("question"), str):
            raise TypeError("fixed local holdout contains a non-string question")
        if not isinstance(sample.get("answer"), list) or not sample["answer"]:
            raise ValueError(
                "fixed local holdout contains a sample without a non-empty answer list"
            )
    if len(holdout) != EXPECTED_SPLIT_COUNTS["holdout"]:
        raise RuntimeError("fixed local holdout query-count mismatch")
    return holdout, {
        "name": "fixed local holdout",
        "queries": len(holdout),
        "source_sha256": source_hash,
        "split_counts": counts,
        "aggregate_only": True,
        "method_selection_or_tuning": False,
    }


def load_corpus(path: Path) -> list[dict]:
    paths = sorted(
        item for item in path.rglob("*")
        if item.is_file() and item.suffix.lower() == ".json"
    )
    if not paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in paths:
        value = read_json(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)
    document_ids = [str(document.get("id")) for document in documents]
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    return documents


def chunk_corpus(documents: list[dict]) -> list[dict]:
    if CHUNK_SIZE - CHUNK_OVERLAP != CHUNK_STEP or CHUNK_STEP <= 0:
        raise RuntimeError("fixed-window controls changed")
    chunks = []
    for document in documents:
        document_id = str(document["id"])
        passage = document.get("passage")
        if not isinstance(passage, str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        for chunk_index, start in enumerate(range(0, len(passage), CHUNK_STEP)):
            end = min(start + CHUNK_SIZE, len(passage))
            text = passage[start:end]
            chunks.append({
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "text": text,
                "char_start": start,
                "char_end": end,
            })
            if chunks[-1]["text"] != passage[start:end]:
                raise RuntimeError("source-preserving chunk invariant failed")
            if end == len(passage):
                break
    return chunks


def model_metadata(model, model_name: str, declared_revision: str, path: Path) -> dict:
    value = getattr(model.config, "_commit_hash", None)
    config_hash = value.strip() if isinstance(value, str) and value.strip() else None
    if config_hash is None:
        status = "declared-offline-snapshot"
    elif config_hash == declared_revision:
        status = "verified-from-config"
    else:
        raise RuntimeError(
            f"{model_name} config _commit_hash {config_hash!r} does not match "
            f"declared revision {declared_revision!r}"
        )
    return {
        "model_name": model_name,
        "declared_revision": declared_revision,
        "config_commit_hash": config_hash,
        "revision_status": status,
        "local_input_path": str(path),
    }


def load_dense_model() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_PATH, local_files_only=True)
    model = AutoModel.from_pretrained(
        DENSE_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(getattr(model.config, "max_position_embeddings", tokenizer_limit))
    if tokenizer_limit < DENSE_MAX_LENGTH or model_limit < DENSE_MAX_LENGTH:
        raise RuntimeError("local dense model does not support max_length=8192")
    metadata = model_metadata(
        model, DENSE_MODEL_NAME, DENSE_DECLARED_REVISION, DENSE_MODEL_PATH
    )
    model.to("cuda")
    model.eval()
    return {
        "tokenizer": tokenizer,
        "model": model,
        "metadata": metadata,
        "load_seconds": perf_counter() - started,
    }


def load_reranker() -> dict:
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        RERANKER_MODEL_PATH, local_files_only=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(getattr(model.config, "max_position_embeddings", tokenizer_limit))
    if tokenizer_limit < RERANKER_MAX_SEQUENCE_LENGTH or model_limit < RERANKER_MAX_SEQUENCE_LENGTH:
        raise RuntimeError("local reranker does not support max_sequence_length=8192")
    metadata = model_metadata(
        model, RERANKER_MODEL_NAME, RERANKER_DECLARED_REVISION, RERANKER_MODEL_PATH
    )
    model.to("cuda")
    model.eval()
    return {
        "tokenizer": tokenizer,
        "model": model,
        "metadata": metadata,
        "load_seconds": perf_counter() - started,
    }


def encode_normalized_cls(model_bundle: dict, texts: list[str], batch_size: int) -> dict:
    embeddings = []
    started = perf_counter()
    for batch_start in range(0, len(texts), batch_size):
        batch = texts[batch_start:batch_start + batch_size]
        inputs = model_bundle["tokenizer"](
            batch,
            padding=True,
            truncation=True,
            max_length=DENSE_MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            outputs = model_bundle["model"](**inputs, return_dict=True)
            embedding = F.normalize(outputs.last_hidden_state[:, 0], p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid normalized CLS embeddings")
        embeddings.append(embedding.cpu())
    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense embedding count mismatch")
    return {"embeddings": encoded, "seconds": perf_counter() - started}


def aggregate_dense_hits(raw_hits: list[tuple[float, int]], chunks: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    for chunk_rank, (score_value, chunk_index) in enumerate(raw_hits, start=1):
        score = float(score_value)
        if not isfinite(score):
            raise RuntimeError("dense retrieval produced a non-finite score")
        chunk = chunks[int(chunk_index)]
        grouped[chunk["document_id"]].append({
            "chunk_index": int(chunk_index),
            "dense_score": score,
            "chunk_rank": chunk_rank,
        })

    documents = []
    for document_id, hits in grouped.items():
        ordered = sorted(
            hits,
            key=lambda hit: (-hit["dense_score"], hit["chunk_rank"], hit["chunk_index"]),
        )
        documents.append({
            "document_id": document_id,
            "dense_document_score": sum(hit["dense_score"] for hit in ordered[:2]),
            "best_chunk_rank": ordered[0]["chunk_rank"],
            "available_supporting_chunks": len(ordered),
            "supporting_chunk_indices": [
                hit["chunk_index"] for hit in ordered[:max(SUPPORT_POOL_SIZES)]
            ],
            "supporting_dense_scores": [
                hit["dense_score"] for hit in ordered[:max(SUPPORT_POOL_SIZES)]
            ],
        })
    documents.sort(
        key=lambda document: (
            -document["dense_document_score"],
            document["best_chunk_rank"],
            document["document_id"],
        )
    )
    selected = documents[:CANDIDATE_DEPTH]
    if len(selected) != CANDIDATE_DEPTH:
        raise RuntimeError(f"expected {CANDIDATE_DEPTH} candidate documents")
    for original_rank, document in enumerate(selected, start=1):
        document["original_dense_rank"] = original_rank
        expected = min(max(SUPPORT_POOL_SIZES), document["available_supporting_chunks"])
        if len(document["supporting_chunk_indices"]) != expected:
            raise RuntimeError("support-pool construction mismatch")
    ids = [document["document_id"] for document in selected]
    if len(ids) != len(set(ids)):
        raise RuntimeError("candidate ranking contains duplicate document IDs")
    return selected


def retrieve_dense(
    query_embeddings: torch.Tensor,
    corpus_embeddings: torch.Tensor,
    chunks: list[dict],
    sample_ids: list[str],
) -> dict:
    if query_embeddings.shape[0] != len(sample_ids):
        raise ValueError("query embedding count mismatch")
    started = perf_counter()
    corpus_on_gpu = corpus_embeddings.to("cuda")
    candidates = {}
    for batch_start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[batch_start:batch_start + QUERY_BATCH_SIZE]
        query_on_gpu = query_embeddings[
            batch_start:batch_start + len(batch_ids)
        ].to("cuda")
        similarities = query_on_gpu @ corpus_on_gpu.T
        if not torch.isfinite(similarities).all():
            raise RuntimeError("dense similarity produced non-finite scores")
        top_scores, top_indices = torch.topk(
            similarities, k=TOP_K_CHUNKS, dim=1, largest=True, sorted=True
        )
        for row, sample_id in enumerate(batch_ids):
            raw_hits = list(zip(
                top_scores[row].float().cpu().tolist(),
                top_indices[row].cpu().tolist(),
            ))
            raw_hits.sort(key=lambda item: (-item[0], item[1]))
            candidates[sample_id] = aggregate_dense_hits(raw_hits, chunks)
    torch.cuda.synchronize()
    seconds = perf_counter() - started
    del corpus_on_gpu
    torch.cuda.empty_cache()
    rankings = {
        sample_id: [document["document_id"] for document in documents]
        for sample_id, documents in candidates.items()
    }
    return {"candidates": candidates, "rankings": rankings, "seconds": seconds}


def score_pair_records(reranker: dict, records: list[tuple[tuple, str, str]]) -> dict:
    scores = {}
    forward_seconds = 0.0
    total_started = perf_counter()
    for batch_start in range(0, len(records), RERANKER_BATCH_SIZE):
        batch = records[batch_start:batch_start + RERANKER_BATCH_SIZE]
        inputs = reranker["tokenizer"](
            [record[1] for record in batch],
            [record[2] for record in batch],
            padding=True,
            truncation="only_second",
            max_length=RERANKER_MAX_SEQUENCE_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        torch.cuda.synchronize()
        forward_started = perf_counter()
        with torch.no_grad():
            logits = reranker["model"](**inputs, return_dict=True).logits.view(-1).float()
        torch.cuda.synchronize()
        forward_seconds += perf_counter() - forward_started
        values = logits.cpu().tolist()
        if len(values) != len(batch) or not all(isfinite(value) for value in values):
            raise RuntimeError("reranker returned invalid scores")
        for record, value in zip(batch, values):
            key = record[0]
            if key in scores:
                raise RuntimeError("duplicate CE pair key")
            scores[key] = float(value)
    return {
        "scores": scores,
        "pairs": len(records),
        "forward_seconds": forward_seconds,
        "total_seconds": perf_counter() - total_started,
    }


def build_pair_bands(samples: dict, candidates: dict, chunks: list[dict]) -> dict:
    bands = {"positions_1_2": [], "positions_3_4": [], "positions_5_8": []}
    bounds = {
        "positions_1_2": (0, 2),
        "positions_3_4": (2, 4),
        "positions_5_8": (4, 8),
    }
    for sample_id, sample in samples.items():
        for document in candidates[sample_id]:
            indices = document["supporting_chunk_indices"]
            for band_name, (start, stop) in bounds.items():
                for dense_position in range(start, min(stop, len(indices))):
                    chunk_index = indices[dense_position]
                    key = (sample_id, document["document_id"], chunk_index)
                    bands[band_name].append(
                        (key, sample["question"], chunks[chunk_index]["text"])
                    )
    return bands


def score_support_union(reranker: dict, pair_bands: dict) -> dict:
    cache = {}
    diagnostics = {}
    for band_name in ("positions_1_2", "positions_3_4", "positions_5_8"):
        scored = score_pair_records(reranker, pair_bands[band_name])
        if set(cache).intersection(scored["scores"]):
            raise RuntimeError("CE pair bands overlap")
        cache.update(scored["scores"])
        diagnostics[band_name] = {
            "pairs": scored["pairs"],
            "forward_seconds": scored["forward_seconds"],
            "total_seconds": scored["total_seconds"],
        }
    return {"cache": cache, "bands": diagnostics}


def derive_variant(samples: dict, candidates: dict, score_cache: dict, m: int) -> dict:
    rankings = {}
    selected_evidence = {}
    for sample_id in samples:
        ranked_documents = []
        selected_evidence[sample_id] = {}
        for document in candidates[sample_id]:
            support = document["supporting_chunk_indices"][:m]
            if not support:
                raise RuntimeError("candidate document has no supporting chunk")
            scored_chunks = []
            for dense_position, chunk_index in enumerate(support):
                key = (sample_id, document["document_id"], chunk_index)
                if key not in score_cache:
                    raise RuntimeError("missing CE score; no imputation is allowed")
                scored_chunks.append({
                    "chunk_index": chunk_index,
                    "dense_position": dense_position,
                    "ce_score": score_cache[key],
                })
            scored_chunks.sort(
                key=lambda item: (
                    -item["ce_score"], item["dense_position"], item["chunk_index"]
                )
            )
            chosen = scored_chunks[:CE_SELECTED_CHUNKS]
            selected_evidence[sample_id][document["document_id"]] = [
                item["chunk_index"] for item in chosen
            ]
            ranked_documents.append({
                "document_id": document["document_id"],
                "document_ce_score": sum(item["ce_score"] for item in chosen),
                "original_dense_rank": document["original_dense_rank"],
            })
        ranked_documents.sort(
            key=lambda item: (
                -item["document_ce_score"],
                item["original_dense_rank"],
                item["document_id"],
            )
        )
        ranking = [item["document_id"] for item in ranked_documents]
        if len(ranking) != CANDIDATE_DEPTH or len(ranking) != len(set(ranking)):
            raise RuntimeError("variant ranking must contain 100 unique candidate documents")
        rankings[sample_id] = ranking
    return {"rankings": rankings, "selected_evidence": selected_evidence}


def candidate_recall_at_100(samples: dict, rankings: dict) -> float:
    values = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        values.append(len(gold.intersection(rankings[sample_id][:100])) / len(gold))
    return float(np.mean(values))


def make_predictions(rankings: dict) -> dict:
    predictions = {}
    for sample_id, ranked in rankings.items():
        top_ids = [str(document_id) for document_id in ranked[:FINAL_K]]
        if len(top_ids) != FINAL_K or len(top_ids) != len(set(top_ids)):
            raise RuntimeError("top-5 prediction must contain five unique IDs; no deduplication")
        predictions[sample_id] = {"answer": top_ids}
    return predictions


def bundled_scorer_compatible_eval(predictions: dict, truth: dict) -> dict:
    y_pred = {key: value["answer"] for key, value in predictions.items()}
    y_true = {key: value for key, value in truth.items()}
    if set(y_pred) != set(y_true):
        raise RuntimeError("Samples in predictions do not match the reference")
    recall = np.array([
        len(set(y_true[key]) & set(y_pred[key])) / len(y_true[key])
        if 0 < len(y_pred[key]) <= 5 else 0 for key in y_true
    ]).mean()
    precision = np.array([
        len(set(y_true[key]) & set(y_pred[key])) / len(y_pred[key])
        if 0 < len(y_pred[key]) <= 5 else 0 for key in y_pred
    ]).mean()
    return {"precision": float(precision), "recall": float(recall)}


def internal_metrics(samples: dict, rankings: dict) -> dict:
    recalls = {depth: [] for depth in (5, 10, 20, 50, 100)}
    reciprocal_ranks = []
    for sample_id, sample in samples.items():
        ranked = rankings[sample_id]
        if len(ranked) != CANDIDATE_DEPTH or len(ranked) != len(set(ranked)):
            raise RuntimeError("final ranking must contain 100 unique IDs")
        gold = {str(document_id) for document_id in sample["answer"]}
        for depth, values in recalls.items():
            values.append(len(gold.intersection(ranked[:depth])) / len(gold))
        first = next(
            (rank for rank, doc_id in enumerate(ranked, 1) if doc_id in gold), None
        )
        reciprocal_ranks.append(0.0 if first is None else 1.0 / first)
    return {
        **{
            f"recall_at_{depth}": float(np.mean(values))
            for depth, values in recalls.items()
        },
        "mrr": float(np.mean(reciprocal_ranks)),
        "mrr_scope": "fixed top-100; absent gold gives reciprocal rank 0",
    }


def percentile(values: list[int], q: int) -> float | None:
    return None if not values else float(np.percentile(np.asarray(values), q))


def first_gold_summary(samples: dict, rankings: dict) -> dict:
    found = []
    bins = Counter()
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        first = next(
            (
                rank
                for rank, doc_id in enumerate(rankings[sample_id], 1)
                if doc_id in gold
            ),
            None,
        )
        if first is None:
            bins["not_found"] += 1
        else:
            found.append(first)
            label = (
                "rank_1" if first == 1 else "rank_2_5" if first <= 5
                else "rank_6_10" if first <= 10
                else "rank_11_20" if first <= 20
                else "rank_21_50" if first <= 50
                else "rank_51_100"
            )
            bins[label] += 1
    labels = (
        "rank_1", "rank_2_5", "rank_6_10", "rank_11_20",
        "rank_21_50", "rank_51_100", "not_found",
    )
    return {
        "when_found": {
            "median": float(median(found)) if found else None,
            "p90": percentile(found, 90),
            "p95": percentile(found, 95),
        },
        "counts": {label: bins[label] for label in labels},
    }


def support_pool_diagnostics(candidates: dict, query_count: int) -> dict:
    available_counts = [
        document["available_supporting_chunks"]
        for documents in candidates.values()
        for document in documents
    ]
    candidate_documents = len(available_counts)
    if candidate_documents != query_count * CANDIDATE_DEPTH:
        raise RuntimeError("candidate-document diagnostic count mismatch")
    bins = Counter()
    for count in available_counts:
        label = (
            "1" if count == 1 else "2" if count == 2 else "3" if count == 3
            else "4_7" if count <= 7 else "8_plus"
        )
        bins[label] += 1
    by_variant = {}
    for m in SUPPORT_POOL_SIZES:
        pairs = sum(min(m, count) for count in available_counts)
        by_variant[f"m{m}"] = {
            "candidate_documents": candidate_documents,
            "available_supporting_chunks": pairs,
            "total_ce_pairs": pairs,
            "mean_ce_pairs_per_query": pairs / query_count,
            "mean_chunks_per_document": pairs / candidate_documents,
        }
    return {
        "within_original_top_2000_chunk_pool": {
            "candidate_documents": candidate_documents,
            "one_available_dense_chunk": bins["1"],
            "two_available_dense_chunks": bins["2"],
            "three_available_dense_chunks": bins["3"],
            "four_to_seven_available_dense_chunks": bins["4_7"],
            "eight_or_more_available_dense_chunks": bins["8_plus"],
        },
        "by_variant": by_variant,
    }


def evidence_replacement_summary(candidates: dict, selected_evidence: dict) -> dict:
    counts = Counter()
    total = 0
    for sample_id, documents in candidates.items():
        for document in documents:
            dense_top_2 = set(document["supporting_chunk_indices"][:2])
            ce_top_2 = set(selected_evidence[sample_id][document["document_id"]])
            replacements = len(dense_top_2 - ce_top_2)
            if replacements not in (0, 1, 2):
                raise RuntimeError("invalid evidence-replacement count")
            counts[replacements] += 1
            total += 1
    return {
        "candidate_documents": total,
        "fraction_ce_top_2_differs_from_dense_top_2": (
            counts[1] + counts[2]
        ) / total,
        "replacement_counts": {
            "0_replacements": counts[0],
            "1_replacement": counts[1],
            "2_replacements": counts[2],
        },
    }


def metric_deltas(
    control_bundled: dict,
    control_internal: dict,
    alternative_bundled: dict,
    alternative_internal: dict,
) -> dict:
    return {
        "precision": alternative_bundled["precision"] - control_bundled["precision"],
        "recall": alternative_bundled["recall"] - control_bundled["recall"],
        "mrr": alternative_internal["mrr"] - control_internal["mrr"],
        "recall_at_10": (
            alternative_internal["recall_at_10"] - control_internal["recall_at_10"]
        ),
        "recall_at_20": (
            alternative_internal["recall_at_20"] - control_internal["recall_at_20"]
        ),
        "recall_at_50": (
            alternative_internal["recall_at_50"] - control_internal["recall_at_50"]
        ),
        "recall_at_100": (
            alternative_internal["recall_at_100"]
            - control_internal["recall_at_100"]
        ),
    }


def close_to(value: float, expected: float) -> bool:
    return abs(value - expected) <= SANITY_ABS_TOLERANCE


def conclusion_for(deltas: dict) -> str:
    precision_delta = deltas["precision"]
    recall_delta = deltas["recall"]
    if (
        precision_delta > SANITY_ABS_TOLERANCE
        and recall_delta > SANITY_ABS_TOLERANCE
    ):
        return (
            "The DEV-selected m=8 supporting-evidence policy generalizes "
            "directionally on the fixed local holdout."
        )
    if (
        precision_delta < -SANITY_ABS_TOLERANCE
        and recall_delta < -SANITY_ABS_TOLERANCE
    ):
        return (
            "The DEV improvement from the m=8 supporting-evidence policy does not "
            "generalize directionally on the fixed local holdout."
        )
    return (
        "The fixed local holdout shows a precision/recall trade-off for the m=8 "
        "supporting-evidence policy."
    )


In [ ]:
# Run this frozen aggregate-only validation manually on offline Kaggle.
run_started = perf_counter()
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle CUDA accelerator")
torch.cuda.reset_peak_memory_stats()

controls = (
    CHUNK_SIZE,
    CHUNK_OVERLAP,
    CHUNK_STEP,
    TOP_K_CHUNKS,
    DOCUMENT_AGGREGATION,
    CANDIDATE_DEPTH,
    SUPPORT_POOL_SIZES,
    CE_SELECTED_CHUNKS,
    DENSE_MAX_LENGTH,
    RERANKER_MAX_SEQUENCE_LENGTH,
    RERANKER_BATCH_SIZE,
    FINAL_K,
)
expected_controls = (
    2_000,
    200,
    1_800,
    2_000,
    "sum_top_2_dense_chunk_scores",
    100,
    (2, 8),
    2,
    8_192,
    8_192,
    128,
    5,
)
if controls != expected_controls:
    raise RuntimeError("a frozen holdout control changed; stop")

# The loader returns only the fixed local holdout after hash and split-count checks.
# There is no DEV method selection, tuning, public inference, or sample-level output.
holdout_samples, split_info = load_fixed_holdout(LEGALIR_SOURCE_PATH)
documents = load_corpus(CORPUS_PATH)
chunks = chunk_corpus(documents)
if len(documents) != EXPECTED_DOCUMENTS or len(chunks) != EXPECTED_CHUNKS:
    raise RuntimeError(
        f"fixed corpus mismatch: got {len(documents)} documents / {len(chunks)} chunks"
    )

# Retrieve once and construct the exact same dense top-100 candidates for both systems.
dense_model = load_dense_model()
dense_model_load_seconds = dense_model["load_seconds"]
dense_model_metadata = dict(dense_model["metadata"])
corpus_encoding = encode_normalized_cls(
    dense_model, [chunk["text"] for chunk in chunks], CORPUS_BATCH_SIZE
)
corpus_embeddings = corpus_encoding["embeddings"]
dense_corpus_encoding_seconds = corpus_encoding["seconds"]
if not torch.allclose(
    torch.linalg.vector_norm(corpus_embeddings.float(), dim=1),
    torch.ones(len(corpus_embeddings)),
    atol=2e-3,
    rtol=0,
):
    raise RuntimeError("corpus embeddings are not L2-normalized")
query_encoding = encode_normalized_cls(
    dense_model,
    [sample["question"] for sample in holdout_samples.values()],
    QUERY_BATCH_SIZE,
)
dense_query_encoding_seconds = query_encoding["seconds"]
dense_retrieval = retrieve_dense(
    query_encoding["embeddings"],
    corpus_embeddings,
    chunks,
    list(holdout_samples),
)
candidate_recall = candidate_recall_at_100(
    holdout_samples, dense_retrieval["rankings"]
)
if not close_to(candidate_recall, EXPECTED_DENSE_RECALL_AT_100):
    raise RuntimeError(
        "historical dense holdout candidate Recall@100 sanity mismatch: "
        f"expected {EXPECTED_DENSE_RECALL_AT_100}, got {candidate_recall}; stop"
    )

dense_model["model"].to("cpu")
del dense_model, corpus_embeddings, corpus_encoding, query_encoding
gc.collect()
torch.cuda.empty_cache()

# Collect each document's top-8 prefix only from the original top-2000 dense hit pool,
# score the union once, then derive m=2 and m=8 from the same cached CE scores.
support_availability = support_pool_diagnostics(
    dense_retrieval["candidates"], len(holdout_samples)
)
reranker = load_reranker()
pair_bands = build_pair_bands(
    holdout_samples, dense_retrieval["candidates"], chunks
)
union_scoring = score_support_union(reranker, pair_bands)
del pair_bands
score_cache = union_scoring["cache"]
expected_union_pairs = support_availability["by_variant"]["m8"]["total_ce_pairs"]
if len(score_cache) != expected_union_pairs:
    raise RuntimeError("union CE score-cache size mismatch")

# System A must reproduce the historical validated m=2 holdout before m=8 is derived.
truth = {
    sample_id: sample["answer"]
    for sample_id, sample in holdout_samples.items()
}
m2 = derive_variant(
    holdout_samples, dense_retrieval["candidates"], score_cache, 2
)
m2_bundled = bundled_scorer_compatible_eval(
    make_predictions(m2["rankings"]), truth
)
m2_internal = internal_metrics(holdout_samples, m2["rankings"])
historical_m2_checks = {
    "precision": (m2_bundled["precision"], EXPECTED_M2_PRECISION),
    "recall": (m2_bundled["recall"], EXPECTED_M2_RECALL),
    "mrr": (m2_internal["mrr"], EXPECTED_M2_MRR),
    "recall_at_100": (
        m2_internal["recall_at_100"], EXPECTED_DENSE_RECALL_AT_100
    ),
}
if not all(
    close_to(actual, expected)
    for actual, expected in historical_m2_checks.values()
):
    actual = {
        name: value[0] for name, value in historical_m2_checks.items()
    }
    expected = {
        name: value[1] for name, value in historical_m2_checks.items()
    }
    raise RuntimeError(
        "historical m=2 fixed-holdout sanity mismatch: "
        f"expected {expected}, got {actual}; stop before deriving or interpreting m=8"
    )
del m2["selected_evidence"]

# System B is the one frozen DEV-selected change; no other m value is evaluated.
m8 = derive_variant(
    holdout_samples, dense_retrieval["candidates"], score_cache, 8
)
for sample_id in holdout_samples:
    retrieved_ids = frozenset(dense_retrieval["rankings"][sample_id])
    if frozenset(m2["rankings"][sample_id]) != retrieved_ids:
        raise RuntimeError("m=2 changed the fixed dense top-100 candidate IDs")
    if frozenset(m8["rankings"][sample_id]) != retrieved_ids:
        raise RuntimeError("m=8 changed the fixed dense top-100 candidate IDs")
    if frozenset(m2["rankings"][sample_id]) != frozenset(m8["rankings"][sample_id]):
        raise RuntimeError(
            "m=2 and m=8 candidate IDs differ query-by-query; implementation bug"
        )

m8_bundled = bundled_scorer_compatible_eval(
    make_predictions(m8["rankings"]), truth
)
m8_internal = internal_metrics(holdout_samples, m8["rankings"])
if not close_to(m8_internal["recall_at_100"], EXPECTED_DENSE_RECALL_AT_100):
    raise RuntimeError("m=8 Recall@100 sanity mismatch")
if m2_internal["recall_at_100"] != m8_internal["recall_at_100"]:
    raise RuntimeError("Recall@100 differs across m=2 and m=8; implementation bug")

deltas = metric_deltas(
    m2_bundled, m2_internal, m8_bundled, m8_internal
)
if deltas["recall_at_100"] != 0.0:
    raise RuntimeError("Recall@100 delta must be exactly zero")

# Optional diagnostic is aggregated over all candidate documents; no cases are emitted.
evidence_replacement = evidence_replacement_summary(
    dense_retrieval["candidates"], m8["selected_evidence"]
)

result = {
    "split": split_info,
    "controls": {
        "documents": len(documents),
        "chunks": len(chunks),
        "chunk_size_characters": CHUNK_SIZE,
        "overlap_characters": CHUNK_OVERLAP,
        "step_characters": CHUNK_STEP,
        "source_preserving_fixed_windows": True,
        "dense_top_k_chunks": TOP_K_CHUNKS,
        "dense_document_aggregation": "sum of top two dense chunk scores",
        "candidate_documents": CANDIDATE_DEPTH,
        "frozen_comparison": ["m2", "m8"],
        "support_source": (
            "per-document prefixes from the original fixed dense top-2000 chunk pool"
        ),
        "cross_encoder_selection": "top two chunks by independent CE score",
        "cross_encoder_document_aggregation": "sum of up to two CE chunk scores",
        "final_k": FINAL_K,
        "fusion": None,
        "title_enrichment": False,
        "article_aware_chunking": False,
        "additional_document_chunk_search": False,
        "aggregate_only_holdout": True,
        "method_selection_or_tuning": False,
    },
    "models": {
        "dense": {
            **dense_model_metadata,
            "representation": "outputs.last_hidden_state[:, 0], then L2 normalization",
            "similarity": "dot product",
            "query_instruction": None,
            "max_length": DENSE_MAX_LENGTH,
            "dynamic_padding": True,
            "dtype": "float16",
            "corpus_batch_size": CORPUS_BATCH_SIZE,
            "query_batch_size": QUERY_BATCH_SIZE,
            "device": "cuda",
            "local_files_only": True,
        },
        "reranker": {
            **reranker["metadata"],
            "pair": "(question, supporting_chunk)",
            "max_sequence_length": RERANKER_MAX_SEQUENCE_LENGTH,
            "dtype": "float16",
            "batch_size": RERANKER_BATCH_SIZE,
            "device": "cuda",
            "local_files_only": True,
        },
    },
    "candidate_invariants": {
        "constructed_once_per_query": True,
        "same_top_100_document_ids_m2_m8": True,
        "candidate_recall_at_100": candidate_recall,
        "expected_historical_candidate_recall_at_100": (
            EXPECTED_DENSE_RECALL_AT_100
        ),
        "recall_at_100_identical": True,
    },
    "m2": {
        "status": "current validated reference; historical sanity reproduced",
        "bundled_scorer": m2_bundled,
        "internal": m2_internal,
        "first_gold_rank": first_gold_summary(
            holdout_samples, m2["rankings"]
        ),
        "support_pool": support_availability["by_variant"]["m2"],
    },
    "m8": {
        "status": "frozen DEV-selected candidate under holdout validation",
        "bundled_scorer": m8_bundled,
        "internal": m8_internal,
        "first_gold_rank": first_gold_summary(
            holdout_samples, m8["rankings"]
        ),
        "support_pool": support_availability["by_variant"]["m8"],
        "aggregate_evidence_replacement": evidence_replacement,
    },
    "deltas": {
        "m8_minus_m2": deltas,
    },
    "support_pool_availability": support_availability,
    "runtime": {
        "dense_model_load_seconds": dense_model_load_seconds,
        "dense_corpus_encoding_seconds": dense_corpus_encoding_seconds,
        "dense_query_encoding_seconds": dense_query_encoding_seconds,
        "dense_retrieval_seconds": dense_retrieval["seconds"],
        "reranker_model_load_seconds": reranker["load_seconds"],
        "ce_scoring_strategy": "score all top-8-required pairs once and cache",
        "ce_pair_bands": union_scoring["bands"],
        "ce_pairs_scored_once": len(score_cache),
        "total_runtime_seconds": perf_counter() - run_started,
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
    },
    "conclusion": conclusion_for(deltas),
}
RESULT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(result, ensure_ascii=False, indent=2))
print("Saved aggregate-only fixed-local-holdout result:", RESULT_PATH)
